# 01b - Pool list

Notebook 01 picks the protocols. This one picks the actual pools we want APY and TVL history for.

Each row is one DeFiLlama pool. The `pool_id` comes from `https://yields.llama.fi/pools` and notebook 02b uses it to hit `https://yields.llama.fi/chart/{pool_id}` for daily history.

We widen the scope a bit so we have something to compare staking against. Beyond Lido and Rocket Pool we add a few lending pools (Aave, Compound) and a few stablecoin-yield pools (Ethena, Sky, Spark).

Coverage:
- Assets: ETH (incl. WETH, stETH, rETH), USDC, USDT, DAI, USDe/sUSDe, USDS/sUSDS
- Chains: Ethereum, Arbitrum, Base
- Categories: staking, lending, stablecoin_yield

Output: `data/processed/pool_mapping.csv` (one row per pool).

In [ ]:
import pandas as pd
from pathlib import Path

## Pool list

How we picked them:
- For each category, just enough pools to be representative, not every pool that exists.
- Lending: Aave V3 on Ethereum, Arbitrum and Base for USDC and WETH, plus USDT and DAI on Ethereum. Compound V3 on Ethereum as a benchmark.
- Stablecoin yields: Ethena sUSDe, Sky's sDAI and sUSDS, plus Spark Savings on Arbitrum and Base.
- Staking: Lido stETH and Rocket Pool rETH. Native ETH is handled separately as a baseline.

In [ ]:
pools = [
    # Staking pools (with yield page on DeFiLlama)
    {"category": "staking",           "project": "lido",          "chain": "Ethereum", "asset": "ETH",  "symbol": "STETH", "pool_id": "747c1d2a-c668-4682-b9f9-296708a3dd90"},
    {"category": "staking",           "project": "rocket-pool",   "chain": "Ethereum", "asset": "ETH",  "symbol": "RETH",  "pool_id": "d4b3c522-6127-4b89-bedf-83641cdcd2eb"},

    # Lending pools: Aave V3
    {"category": "lending",           "project": "aave-v3",       "chain": "Ethereum", "asset": "USDC", "symbol": "USDC",  "pool_id": "aa70268e-4b52-42bf-a116-608b370f9501"},
    {"category": "lending",           "project": "aave-v3",       "chain": "Ethereum", "asset": "USDT", "symbol": "USDT",  "pool_id": "f981a304-bb6c-45b8-b0c5-fd2f515ad23a"},
    {"category": "lending",           "project": "aave-v3",       "chain": "Ethereum", "asset": "DAI",  "symbol": "DAI",   "pool_id": "3665ee7e-6c5d-49d9-abb7-c47ab5d9d4ac"},
    {"category": "lending",           "project": "aave-v3",       "chain": "Ethereum", "asset": "ETH",  "symbol": "WETH",  "pool_id": "e880e828-ca59-4ec6-8d4f-27182a4dc23d"},
    {"category": "lending",           "project": "aave-v3",       "chain": "Arbitrum", "asset": "USDC", "symbol": "USDC",  "pool_id": "d9fa8e14-0447-4207-9ae8-7810199dfa1f"},
    {"category": "lending",           "project": "aave-v3",       "chain": "Arbitrum", "asset": "DAI",  "symbol": "DAI",   "pool_id": "a8e3d841-2788-4647-ad54-5a36fac451b1"},
    {"category": "lending",           "project": "aave-v3",       "chain": "Arbitrum", "asset": "ETH",  "symbol": "WETH",  "pool_id": "e302de4d-952e-4e18-9749-0a9dc86e98bc"},
    {"category": "lending",           "project": "aave-v3",       "chain": "Base",     "asset": "USDC", "symbol": "USDC",  "pool_id": "7e0661bf-8cf3-45e6-9424-31916d4c7b84"},
    {"category": "lending",           "project": "aave-v3",       "chain": "Base",     "asset": "ETH",  "symbol": "WETH",  "pool_id": "23405eee-97e7-4b8e-8625-19c3a36047e8"},

    # Lending pools: Compound V3 (Ethereum benchmark)
    {"category": "lending",           "project": "compound-v3",   "chain": "Ethereum", "asset": "USDC", "symbol": "USDC",  "pool_id": "7da72d09-56ca-4ec5-a45f-59114353e487"},
    {"category": "lending",           "project": "compound-v3",   "chain": "Ethereum", "asset": "USDT", "symbol": "USDT",  "pool_id": "f4d5b566-e815-4ca2-bb07-7bcd8bc797f1"},
    {"category": "lending",           "project": "compound-v3",   "chain": "Ethereum", "asset": "ETH",  "symbol": "ETH",   "pool_id": "85c57261-b75b-4447-a115-d79b1a7de8ed"},

    # Stablecoin yield pools
    {"category": "stablecoin_yield",  "project": "ethena-usde",   "chain": "Ethereum", "asset": "USDe", "symbol": "SUSDE", "pool_id": "66985a81-9c51-46ca-9977-42b4fe7bc6df"},
    {"category": "stablecoin_yield",  "project": "sky-lending",   "chain": "Ethereum", "asset": "DAI",  "symbol": "SDAI",  "pool_id": "c8a24fee-ec00-4f38-86c0-9f6daebc4225"},
    {"category": "stablecoin_yield",  "project": "sky-lending",   "chain": "Ethereum", "asset": "USDS", "symbol": "SUSDS", "pool_id": "d8c4eff5-c8a9-46fc-a888-057c4c668e72"},
    {"category": "stablecoin_yield",  "project": "spark-savings", "chain": "Arbitrum", "asset": "USDS", "symbol": "USDS",  "pool_id": "9d499222-a01a-45bb-bbc9-f01c7923693b"},
    {"category": "stablecoin_yield",  "project": "spark-savings", "chain": "Base",     "asset": "USDS", "symbol": "USDS",  "pool_id": "aa2d08c0-0abd-4dcf-be93-ff8ca89d01cd"},
]

pool_mapping_df = pd.DataFrame(pools)
print(f"Total pools: {len(pool_mapping_df)}")
print(pool_mapping_df.groupby(["category", "chain"]).size().unstack(fill_value=0))
pool_mapping_df

## Save

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

pool_mapping_df.to_csv(DATA_PROCESSED / "pool_mapping.csv", index=False)
print("Saved:", DATA_PROCESSED / "pool_mapping.csv")